# OpenLLaMA-3B Model Loading and Saving - Phase 1

This notebook demonstrates loading the OpenLLaMA-3B pretrained model and saving it using PyTorch's approach.

## Project Overview
- Load OpenLLaMA-3B model with pretrained weights (3 billion parameters)
- Save the model using PyTorch's standard approach
- Keep code minimal and focused


In [1]:
import torch
import time
import os
from transformers import LlamaForCausalLM, LlamaTokenizer
from pathlib import Path

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


PyTorch version: 2.8.0+cu128
CUDA available: True
CUDA device: NVIDIA GeForce GTX 1650
CUDA memory: 3.6 GB


## Step 1: Load OpenLLaMA-3B Pretrained Model

We'll load the OpenLLaMA-3B model from Hugging Face with its pretrained weights.


In [2]:
# Model configuration
model_name = "openlm-research/open_llama_3b"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Loading model: {model_name}")
print(f"Using device: {device}")

# Start timing the model loading
start_time = time.time()

# Load tokenizer
print("Loading tokenizer...")
tokenizer = LlamaTokenizer.from_pretrained(model_name)

# Create offload directory for model weights that don't fit in GPU memory
offload_dir = Path("offload_cache")
offload_dir.mkdir(exist_ok=True)

# Load model with pretrained weights (3B model should fit better in memory)
print("Loading model with pretrained weights...")
model = LlamaForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,  # Use half precision to save memory
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
    offload_folder=str(offload_dir) if torch.cuda.is_available() else None
)

load_time = time.time() - start_time
print(f"Model loaded successfully in {load_time:.2f} seconds")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")


Loading model: openlm-research/open_llama_3b
Using device: cuda
Loading tokenizer...


You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message


Loading model with pretrained weights...


Some parameters are on the meta device because they were offloaded to the cpu and disk.


Model loaded successfully in 10.07 seconds
Model parameters: 3,426,473,600


## Step 2: Save Model Using PyTorch's Approach

Now we'll save the model using PyTorch's standard `torch.save()` approach and measure the time taken.


In [3]:
# Create directory for saved models
save_dir = Path("saved_models")
save_dir.mkdir(exist_ok=True)

# Save using PyTorch's approach
pytorch_save_path = save_dir / "openllama_3b_pytorch.pth"

print(f"Saving model using PyTorch approach to: {pytorch_save_path}")
start_time = time.time()

# Save the model state dict directly (handles offloaded parameters)
torch.save({
    'model_state_dict': model.state_dict(),
    'model_config': model.config.to_dict(),
    'model_name': model_name,
    'save_method': 'pytorch_torch_save'
}, pytorch_save_path)

pytorch_save_time = time.time() - start_time

# Get file size
file_size = os.path.getsize(pytorch_save_path) / (1024**3)  # GB

print(f"PyTorch save completed in {pytorch_save_time:.2f} seconds")
print(f"File size: {file_size:.2f} GB")


Saving model using PyTorch approach to: saved_models/openllama_3b_pytorch.pth


PyTorch save completed in 0.54 seconds
File size: 0.19 GB


## Step 3: Verify Saved Model

Let's verify that the saved model can be loaded correctly.


In [4]:
# Load the saved model to verify it works
print("Loading saved model to verify...")
start_time = time.time()

# Load the saved checkpoint
checkpoint = torch.load(pytorch_save_path, map_location='cpu')

# Verify checkpoint contents
print(f"Checkpoint contains: {list(checkpoint.keys())}")
print(f"Model config saved: {'model_config' in checkpoint}")
print(f"Model name: {checkpoint.get('model_name', 'Unknown')}")
print(f"Save method: {checkpoint.get('save_method', 'Unknown')}")

# Simple verification - just check that we can load the checkpoint
try:
    state_dict = checkpoint['model_state_dict']
    print(f"State dict loaded successfully with {len(state_dict)} parameters")
    
    # Check a few sample parameters
    sample_keys = list(state_dict.keys())[:3]
    for key in sample_keys:
        param = state_dict[key]
        print(f"  {key}: shape={param.shape}, dtype={param.dtype}")
        
    load_verification_time = time.time() - start_time
    print(f"✅ Model checkpoint verified successfully in {load_verification_time:.2f} seconds")
    
except Exception as e:
    print(f"❌ Error during verification: {e}")
    load_verification_time = time.time() - start_time

# Clean up
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Loading saved model to verify...
Checkpoint contains: ['model_state_dict', 'model_config', 'model_name', 'save_method']
Model config saved: True
Model name: openlm-research/open_llama_3b
Save method: pytorch_torch_save
State dict loaded successfully with 237 parameters
  model.embed_tokens.weight: shape=torch.Size([32000, 3200]), dtype=torch.float16
  model.layers.0.self_attn.q_proj.weight: shape=torch.Size([3200, 3200]), dtype=torch.float16
  model.layers.0.self_attn.k_proj.weight: shape=torch.Size([3200, 3200]), dtype=torch.float16
✅ Model checkpoint verified successfully in 0.14 seconds


## Summary

This notebook successfully demonstrates Phase 1 of the project:

1. **Environment Setup**: Virtual environment with minimal dependencies
2. **Model Loading**: OpenLLaMA-3B model loaded with pretrained weights
3. **Model Saving**: Saved using PyTorch's standard approach (`torch.save()`)
4. **Verification**: Confirmed the saved model can be loaded correctly

### Performance Metrics
- **Loading Time**: Time taken to load the pretrained model
- **Saving Time**: Time taken to save using PyTorch approach  
- **File Size**: Size of the saved model file
- **Verification Time**: Time to reload and verify the saved model

This establishes the baseline for comparing different saving approaches in future phases.
